# <a href="https://girafe.ai/" target="_blank" rel="noopener noreferrer"><img src="https://raw.githubusercontent.com/girafe-ai/ml-course/7096a5df4cada5ee651be1e3215c2f7fb8a7e0bf/logo_margin.svg" alt="girafe-ai logo" width="150px" align="left"></a> [ml-basic course](https://github.com/girafe-ai/ml-course) <a class="tocSkip">

# Day 07: kNN indexes

In [1]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 41.2 MB/s eta 0:00:00


In [3]:
from tqdm import tqdm, tnrange, tqdm_notebook
import numpy as np
import os, sys
import matplotlib.pyplot as plt
import pickle
import json
import gdown
import h5py
import faiss

## 1. Load embeddings

In [4]:
file_id = '1-1t1sbQXXyB-qZr-czqv2UJ4E5BQ7McI'
destination = 'lastftm_factors.h5'
download_url = f"https://drive.google.com/uc?id={file_id}"

gdown.download(download_url, destination, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1-1t1sbQXXyB-qZr-czqv2UJ4E5BQ7McI
From (redirected): https://drive.google.com/uc?id=1-1t1sbQXXyB-qZr-czqv2UJ4E5BQ7McI&confirm=t&uuid=fecc1e36-fb88-42de-ab21-ca47c87b4d3e
To: /content/lastftm_factors.h5
100%|██████████| 333M/333M [00:06<00:00, 48.5MB/s]


'lastftm_factors.h5'

In [5]:
h5f = h5py.File('/content/lastftm_factors.h5', 'r')
item_factors = h5f['items'][:]
user_factors = h5f['users'][:]
h5f.close()

In [6]:
item_factors.shape

(292385, 128)

In [7]:
user_factors.shape

(358868, 128)

## 2. Neighbours search with FAISS

[Faiss docs](https://faiss.ai/)

$<a, b> = \sum(a_i * b_i)$

In [10]:
%%time
# Create index Exact Search for Inner Product
ip_index = faiss.IndexFlatIP(item_factors.shape[1])

# Add document embeddings to index
ip_index.add(item_factors)
print(ip_index.ntotal)

292385
CPU times: user 51.2 ms, sys: 74.5 ms, total: 126 ms
Wall time: 127 ms


In [11]:
k = 50  # Choose 50 nearest neighbours
n_users = 100  # 100 users is real example of request batch

user_embeddings = user_factors[:n_users]

In [12]:
%%timeit
# D - distances
# I - indices
dist_brute, index_brute = ip_index.search(user_embeddings, k)

207 ms ± 8.6 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [13]:
dist_brute, index_brute = ip_index.search(user_embeddings, k)

In [14]:
dist_brute.shape, index_brute.shape

((100, 50), (100, 50))

In [15]:
dist_brute[:10]

array([[1.1410543 , 1.1322321 , 1.1187365 , 1.0928438 , 1.0664618 ,
        1.062733  , 1.05291   , 1.0320057 , 1.0301973 , 1.0179089 ,
        1.0020028 , 0.9837904 , 0.98047704, 0.9796851 , 0.96542853,
        0.9620888 , 0.95902544, 0.9556296 , 0.95552677, 0.95469624,
        0.95193106, 0.95178026, 0.9506871 , 0.9500801 , 0.94741666,
        0.9468063 , 0.93974173, 0.92981064, 0.9231755 , 0.92246383,
        0.91976446, 0.9178443 , 0.91715044, 0.9113436 , 0.9098235 ,
        0.9073585 , 0.90652895, 0.9046614 , 0.90354747, 0.90326846,
        0.89783937, 0.89204746, 0.891694  , 0.8910956 , 0.8886769 ,
        0.88656265, 0.8863779 , 0.88404644, 0.8812577 , 0.8803741 ],
       [1.2508918 , 1.249527  , 1.2257144 , 1.2112938 , 1.1923635 ,
        1.189894  , 1.1788456 , 1.1644775 , 1.1519464 , 1.1518093 ,
        1.1348926 , 1.1333601 , 1.131854  , 1.1168416 , 1.1018484 ,
        1.0981128 , 1.0895374 , 1.088695  , 1.0817112 , 1.0800222 ,
        1.0795983 , 1.0732406 , 1.0706415 , 1.0

In [16]:
index_brute[:10]

array([[150177, 161850, 107119, 144310,  76757, 255141, 155259,  46258,
        136336, 259874, 255068, 275919, 255226, 217076,  90705, 137079,
        254099, 165938, 155258, 199287,   4580,  35023, 111603, 272757,
        185384, 165137, 116827, 230026,  37529,  45575, 140710, 249555,
          3416,  98993, 205631, 164039, 201166,  50610, 219096, 168138,
        249560, 129399, 228777,  48292, 175819, 252787, 110905, 168137,
        245502,  12537],
       [186566,  86649,  71465,  78295, 229823,  54967, 189597, 252956,
        250342, 267949,  71225,  86665,  50365, 155806, 255779, 262990,
        125452, 261137, 274818, 208011, 128505, 248849,  26278, 214351,
        235136, 219131, 116919,  32593,  56783, 186441, 111764,  27873,
        176605,  56308, 117795, 257566, 242523, 190802,  75756, 263348,
        176709,  53756,  33246,  53846,  68513, 258362,  54515, 262555,
        266344, 192736],
       [180940, 142885, 120981,  37629,  78810, 166647, 159270, 234769,
        241312

Let's compare with vanilla method to find nearest neighbours:

In [17]:
# too long because we sort full array
product = user_embeddings.dot(item_factors.T)
exact_indexes = np.argsort(product, axis=1)[:,:-k-1:-1]

In [18]:
%%time
product = user_embeddings.dot(item_factors.T)
neighbours = np.partition(-1 * product, k, axis=1)[:, :k]
exact_distances = -1 * np.sort(neighbours, axis=1)

CPU times: user 540 ms, sys: 128 ms, total: 668 ms
Wall time: 660 ms


In [19]:
np.allclose(dist_brute, exact_distances)

True

In [20]:
np.equal(index_brute, exact_indexes).mean()

np.float64(1.0)

## 3. HNSW

<img src="hsnw.png" width="500">



M is the number of neighbors used in the graph. A larger M is more accurate but uses more memory

efConstruction is the depth of exploration at add time

efSearch is the depth of exploration of the search

**HNSW** (Hierarchical navigable small world) - graph-based approximate nearest neighbor algorithm.

In [21]:
%%time
neighbours = 50


hnsw_index = faiss.IndexHNSWFlat(item_factors.shape[1], neighbours, faiss.METRIC_INNER_PRODUCT) # Create HNSW index

hnsw_index.hnsw.efSearch = 1600
hnsw_index.hnsw.m = 96
hnsw_index.hnsw.efConstruction = 1000

faiss.normalize_L2(item_factors)
hnsw_index.add(item_factors)

CPU times: user 20min 49s, sys: 2.07 s, total: 20min 51s
Wall time: 12min 3s


In [24]:
%%timeit
dist_hnsw, index_hnsw = hnsw_index.search(user_embeddings, k)

407 ms ± 41.9 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [23]:
index_hnsw[:1]

array([[163197, 251773,  18748, 203356, 259874, 112589, 201943,  45575,
        275850, 288601,  62309,   5322,  70526, 253506, 138228,  65428,
        236277,  14673, 145233, 254099,   4580, 164482, 262401,  85321,
        186493, 249989,  63381, 111282,  59325, 186026,  98049, 151174,
        254338,  41017,  91144,  32782,  35176, 283940,  86698, 272362,
         99683, 171267,  50610, 158124, 201166,  45568, 105929, 161501,
        206361,   8618]])

In [ ]:
dist_hnsw[:1]

array([[9.9573345, 9.653416 , 9.609156 , 9.540543 , 9.387692 , 9.332269 ,
        9.27276  , 9.258057 , 9.225113 , 9.214386 , 9.173318 , 8.954049 ,
        8.887032 , 8.864248 , 8.8621025, 8.792911 , 8.771091 , 8.767221 ,
        8.73971  , 8.651405 , 8.616867 , 8.601856 , 8.596088 , 8.523359 ,
        8.512981 , 8.511211 , 8.505212 , 8.464906 , 8.450711 , 8.438483 ,
        8.370792 , 8.305387 , 8.3022   , 8.250093 , 8.237274 , 8.203123 ,
        8.162453 , 8.142109 , 8.140543 , 8.110386 , 8.069276 , 8.034338 ,
        8.027824 , 8.025734 , 7.978615 , 7.918512 , 7.9149528, 7.913948 ,
        7.910551 , 7.8855906]], dtype=float32)

In [ ]:
%%time
product = user_embeddings.dot(item_factors.T)
exact_indexes = np.argsort(product, axis=1)[:,:-k-1:-1]

CPU times: user 2.08 s, sys: 70.6 ms, total: 2.15 s
Wall time: 1.9 s


In [ ]:
np.equal(index_hnsw, exact_indexes).mean()

np.float64(0.9736)

Algorithm is not tuned for the dataset, so results are not the best. But you can find optimal parameters for 99% or even 99.9%.

## Extra: Nearest neighbours using KDTree

In [ ]:
from sklearn.neighbors import KDTree

In [ ]:
k = 50

tree = KDTree(item_factors, leaf_size=5)
kdtree_dists, kdtree_indexes = tree.query(user_embeddings, k=k)
print(kdtree_indexes)

[[163197 251773  18748 ... 161501 206361   8618]
 [187610 246138  13544 ... 184715 261870 140707]
 [133871 223203  24014 ...  85163 196410 117829]
 ...
 [139982  72580 166334 ... 222646  50417  25495]
 [207501  51965 151438 ...  44147  24664 158761]
 [186242  45911  96262 ... 266535  33604  94169]]


In [ ]:
np.equal(kdtree_indexes, exact_indexes).mean()

np.float64(1.0)